# リズム天国 PC版：声と楽器の AI 復元

`tools/samplematch/ai_prepare.py` で作った `rtpc_ai_input.zip` をアップロードして、
声の復元（VoiceFixer・Resemble Enhance）と高音の補完（AudioSR）をかけます。
結果は `rtpc_ai_output.zip` としてダウンロードされ、Mac 側の `ai_import.py` で取り込みます。

**使い方**: 「ランタイム → ランタイムのタイプを変更」で **T4 GPU** を選び、上から順に実行してください。
各 AI は別々の Python 環境に入れるので、互いの部品がぶつかりません。
時間のかかる AudioSR は、最初は `LIMIT` で数を絞って試すのがおすすめです。

※ 2026-10-21 の初回実行で出た 3 つのエラー（matplotlib の設定、git-lfs、pkg_resources）を直した版です。

In [ ]:
# 0. What to run.
#    VoiceFixer rebuilds the voice through a vocoder: thin, high voices can
#    gain a low end, fuller ones come out muffled or hoarse. Resemble
#    Enhance is the other voice model. AudioSR only adds treble above the
#    original. AUDIOSR_LIMIT = -1 skips it, 0 = every input (~9 s each).
RUN_VOICEFIXER = False
RUN_RESEMBLE = True
RESEMBLE_LIMIT = 0      # 0 = all 508 voice inputs
AUDIOSR_LIMIT = -1

In [ ]:
# 1. Upload rtpc_ai_input.zip and unpack it (skipped if already unpacked,
#    so "run all" again does not ask for the file a second time)
import zipfile, os
if not os.path.isdir('/content/in/voice'):
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    zipfile.ZipFile(name).extractall('/content/in')
print(len(os.listdir('/content/in/voice')), 'voice inputs,', len(os.listdir('/content/in/inst')), 'instrument inputs')

In [ ]:
# 2. uv, to give each model its own environment; git-lfs, which Resemble
#    Enhance uses to fetch its model
!pip -q install uv
!apt-get -qq install -y git-lfs > /dev/null && git lfs install > /dev/null
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 声: VoiceFixer

In [ ]:
%%writefile /content/run_voicefixer.py
import os, sys, glob
from voicefixer import VoiceFixer
vf = VoiceFixer()
os.makedirs('/content/out/voicefixer', exist_ok=True)
files = sorted(glob.glob('/content/in/voice/*.wav'))
for i, f in enumerate(files):
    out = os.path.join('/content/out/voicefixer', os.path.basename(f))
    if not os.path.exists(out):
        vf.restore(input=f, output=out, cuda=True, mode=0)
    if i % 25 == 0:
        print(i, '/', len(files), flush=True)
print('done')

In [ ]:
# MPLBACKEND: Colab's inline backend is not installed in the model's own
# environment, and matplotlib refuses to start with it.
if RUN_VOICEFIXER:
    !uv venv -q --clear --python 3.10 /content/env_vf
    !uv pip install -q --python /content/env_vf/bin/python voicefixer
    !MPLBACKEND=Agg /content/env_vf/bin/python /content/run_voicefixer.py

## 声: Resemble Enhance

In [ ]:
%%writefile /content/run_resemble.py
# Resemble Enhance resamples its input to 44.1 kHz with torchaudio, whose
# polyphase filter has orig/gcd phases: 13379 Hz shares no factor with
# 44100, so building it took most of the ~30 s each file needed (about two
# files a minute on the second run; the model itself takes ~1.3 s). Hand
# it 44.1 kHz instead, resampled here by FFT. Files are read and written
# with soundfile, not torchaudio.
import os, glob, traceback, numpy as np, soundfile as sf, torch
from resemble_enhance.enhancer.inference import enhance

RATE = 44100
os.makedirs('/content/out/resemble', exist_ok=True)
files = sorted(glob.glob('/content/in/voice/*.wav'))
limit = int(open('/content/resemble_limit.txt').read())
if limit:
    files = files[:limit]
for i, f in enumerate(files):
    out = os.path.join('/content/out/resemble', os.path.basename(f))
    if os.path.exists(out):
        continue
    try:
        x, sr = sf.read(f, dtype='float32', always_2d=True)
        x = x.mean(1)
        if sr != RATE:
            n = int(round(len(x) * RATE / sr))
            X = np.fft.rfft(x)
            Y = np.zeros(n // 2 + 1, complex)
            k = min(len(X), len(Y))
            Y[:k] = X[:k]
            x = (np.fft.irfft(Y, n) * (n / len(x))).astype(np.float32)
        hwav, nsr = enhance(torch.from_numpy(x), RATE, 'cuda', nfe=64, solver='midpoint', lambd=0.5, tau=0.5)
        sf.write(out, hwav.cpu().numpy(), nsr)
    except Exception:
        print('failed:', f, flush=True)
        traceback.print_exc()
    if i % 25 == 0:
        print(i, '/', len(files), flush=True)
print('done')

In [ ]:
if RUN_RESEMBLE:
    open('/content/resemble_limit.txt', 'w').write(str(RESEMBLE_LIMIT))
    !mkdir -p /content/out
    !uv venv -q --clear --python 3.10 /content/env_re
    !uv pip install -q --python /content/env_re/bin/python resemble-enhance soundfile
    !MPLBACKEND=Agg /content/env_re/bin/python /content/run_resemble.py 2>&1 | tee /content/out/resemble.log

## 楽器（と声）: AudioSR

1 個あたり 10 秒前後かかります。数は最初のセルの `AUDIOSR_LIMIT` で決めます（0 で全部）。

In [ ]:
open('/content/limit.txt', 'w').write(str(AUDIOSR_LIMIT))

In [ ]:
%%writefile /content/run_audiosr.py
# Builds AudioSR without audiosr.build_model: that loads the whole 6.2 GB
# checkpoint at once (onto the GPU, after building the model in RAM), and
# on the free tier the process vanished without an error -- most likely
# killed for memory. Here the checkpoint is memory-mapped, the EMA copy and
# the CLAP model (never used for super-resolution) are left out, and the
# model is moved to the GPU once.
import os, glob, gc, traceback, numpy as np, soundfile as sf, torch, torch.nn as nn
import torchaudio

# torchaudio 2.9+ hands file reading to torchcodec, which is not installed
# ("Please install torchcodec to use this function" on the fourth run);
# AudioSR only reads plain WAVs, so read them with soundfile.
def _load(path, *a, **k):
    data, sr = sf.read(path, dtype='float32', always_2d=True)
    return torch.from_numpy(data.T.copy()), sr
torchaudio.load = _load
import audiosr.latent_diffusion.models.ddpm as ddpm
from audiosr.utils import default_audioldm_config, download_checkpoint
from audiosr import super_resolution

class NoClap(nn.Module):
    def __init__(self, *a, **k):
        super().__init__()

ddpm.CLAPAudioEmbeddingClassifierFreev2 = NoClap
cfg = default_audioldm_config('basic')
cfg['model']['params']['device'] = 'cuda'
cfg['model']['params']['use_ema'] = False
m = ddpm.LatentDiffusion(**cfg['model']['params'])
sd = torch.load(download_checkpoint('basic'), map_location='cpu', mmap=True, weights_only=False)['state_dict']
sd = {k: v for k, v in sd.items() if not k.startswith(('model_ema', 'clap.'))}
print('load_state_dict:', m.load_state_dict(sd, strict=False), flush=True)
del sd
gc.collect()
m = m.eval().to('cuda')
print('model on GPU, %.1f GB used' % (torch.cuda.memory_allocated() / 1e9), flush=True)

limit = int(open('/content/limit.txt').read())
os.makedirs('/content/out/audiosr', exist_ok=True)
files = sorted(glob.glob('/content/in/inst/*.wav'))
if limit:
    files = files[:limit]
for i, f in enumerate(files):
    out = os.path.join('/content/out/audiosr', os.path.basename(f))
    if os.path.exists(out):
        continue
    try:
        # Hand AudioSR a 48 kHz file. Its own resampler builds a polyphase
        # filter over orig/gcd phases, and 13379 Hz (the GBA's rate) shares
        # no factor with 48000: on the fourth run the first 13379 Hz sample
        # grew the process to 11.6 GB until it was killed. An FFT resample
        # here costs a few MB.
        x, sr = sf.read(f, dtype='float32', always_2d=True)
        x = x.mean(1)
        if sr != 48000:
            n = int(round(len(x) * 48000 / sr))
            X = np.fft.rfft(x)
            Y = np.zeros(n // 2 + 1, complex)
            k = min(len(X), len(Y))
            Y[:k] = X[:k]
            x = (np.fft.irfft(Y, n) * (n / len(x))).astype(np.float32)
        sf.write('/content/_in48.wav', x, 48000)
        w = np.asarray(super_resolution(m, '/content/_in48.wav', seed=42, guidance_scale=3.5, ddim_steps=50), dtype=np.float32).squeeze()
        sf.write(out, w, 48000)
    except Exception:
        print('failed:', f, flush=True)
        traceback.print_exc()
    if i % 10 == 0:
        print(i, '/', len(files), flush=True)
print('done')

In [ ]:
# setuptools<70 and matplotlib: librosa 0.9 still imports pkg_resources and
# audiosr imports matplotlib, neither of which a bare uv environment has.
# Everything, with free memory before and after and the exit status, also
# goes to /content/out/audiosr.log, so the downloaded zip carries the reason
# if it fails.
if AUDIOSR_LIMIT >= 0:
    !mkdir -p /content/out && (free -m; nvidia-smi --query-gpu=memory.used,memory.total --format=csv) > /content/out/audiosr.log 2>&1
    !uv venv -q --clear --python 3.10 /content/env_sr >> /content/out/audiosr.log 2>&1
    !uv pip install -q --python /content/env_sr/bin/python audiosr==0.0.7 soundfile matplotlib "setuptools<70" >> /content/out/audiosr.log 2>&1
    !MPLBACKEND=Agg /content/env_sr/bin/python /content/run_audiosr.py 2>&1 | tee -a /content/out/audiosr.log; echo "exit status ${PIPESTATUS[0]}" | tee -a /content/out/audiosr.log
    !(free -m; dmesg 2>/dev/null | tail -5) >> /content/out/audiosr.log 2>&1

## 結果をダウンロード

In [ ]:
!cd /content/out && ls && zip -q -r /content/rtpc_ai_output.zip .
from google.colab import files
files.download('/content/rtpc_ai_output.zip')